In [1]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph,START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image,display
from langchain_core.tools import Tool
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver
import gradio as gr
import os
import requests


In [2]:
load_dotenv(override=True)

True

In [3]:
class State(TypedDict):
    
    messages: Annotated[list, add_messages]


graph_builder = StateGraph(State)

In [4]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

tool_push = Tool(
        name="send_push_notification",
        func=push,
        description="useful for when you want to send a push notification"
    )

In [5]:
import nest_asyncio
nest_asyncio.apply()

In [6]:
from langchain_community.agent_toolkits import PlayWrightBrowserToolkit
from langchain_community.tools.playwright.utils import create_async_playwright_browser
async_browser =  create_async_playwright_browser(headless=True)  # headful mode
toolkit = PlayWrightBrowserToolkit.from_browser(async_browser=async_browser)
tools = toolkit.get_tools()

In [7]:
for tool in tools:
    print(f"{tool.name}={tool}")

click_element=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/codespace/.cache/ms-playwright/chromium-1208/chrome-linux64/chrome> version=145.0.7632.6>
navigate_browser=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/codespace/.cache/ms-playwright/chromium-1208/chrome-linux64/chrome> version=145.0.7632.6>
previous_webpage=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/codespace/.cache/ms-playwright/chromium-1208/chrome-linux64/chrome> version=145.0.7632.6>
extract_text=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/codespace/.cache/ms-playwright/chromium-1208/chrome-linux64/chrome> version=145.0.7632.6>
extract_hyperlinks=async_browser=<Browser type=<BrowserType name=chromium executable_path=/home/codespace/.cache/ms-playwright/chromium-1208/chrome-linux64/chrome> version=145.0.7632.6>
get_elements=async_browser=<Browser type=<BrowserType name=chromium executable_path=/

In [8]:
tool_dict = {tool.name:tool for tool in tools}
navigate_tool = tool_dict.get("navigate_browser")
extract_text_tool = tool_dict.get("extract_text")

In [9]:
await navigate_tool.arun({"url": "https://www.cnn.com"})
text = await extract_text_tool.arun({})

In [ ]:
all_tools = tools + [tool_push]

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini")
llm_with_tools = llm.bind_tools(all_tools)


def chatbot(state: State):
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

In [ ]:
graph_builder=StateGraph(State)
graph_builder.add_node("chatbot",chatbot)
graph_builder.add_node("tools",ToolNode(tools=all_tools))
graph_builder.add_conditional_edges("chatbot",tools_condition,"tools")
graph_builder.add_edge("tools","chatbot")
graph_builder.add_edge(START,"chatbot")

memory = MemorySaver()
graph = graph_builder.compile(checkpointer=memory)
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
config = {"configurable": {"thread_id": "10"}}

async def chat(user_input: str, history):
    result = await graph.ainvoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat, type="messages").launch()